In [ ]:
import awkward as ak
import numpy as np
import time
import coffea
import uproot
import hist
import vector
import json
print("awkward version ", ak.__version__)
print("coffea version ", coffea.__version__)
from coffea import util, processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema, BaseSchema
from collections import defaultdict
import pickle
import glob
import matplotlib.pyplot as plt

In [ ]:
fileset = {"2017":[]} #{"2016":[], "2016APV":[], "2017":[], "2018":[]}

with open('data/nanoAOD/ZPrime1.json') as json_file:
    data = json.load(json_file)
    # print(data['2016'][0])
    # print(len(data))
    # fileset['2016APV'] += ['root://cmsxrootd/'+data['2016APV']['1000'][i] for i in range(len(data['2016APV']['1000']))]
    # fileset['2016'] += ['root://cmsxrootd/'+data['2016']['1000'][i] for i in range(len(data['2016']['1000']))]
    fileset['2017'] += [data['2017']['1000'][i] for i in range(len(data['2017']['1000']))]
    # fileset['2018'] += [data['2018']['1000'][i] for i in range(len(data['2018']['1000']))]
print(fileset)

In [ ]:
class GetHT(processor.ProcessorABC):
    def __init__(self ):
    
        HT_axis = hist.axis.Regular( 10, 950, 1450, name="HT", label=r"AK4 Jet $H_T$")
        HTfull_axis = hist.axis.Regular( 41, 950, 3000, name="HT", label=r"AK4 Jet $H_T$")
        dataset_axis = hist.axis.StrCategory([], growth=True, name="dataset", label="Primary dataset")
        
        self.hists = {
            "ht":hist.Hist(dataset_axis, HT_axis, storage="weight", label="Counts"),
            "htfull":hist.Hist(dataset_axis, HTfull_axis, storage="weight", label="Counts")
        }
        
        self.means_stddevs = defaultdict()
    
    @property
    def accumulator(self):
        return self.hists

    
    # we will receive a NanoEvents instead of a coffea DataFrame
    def process(self, events):
        dataset = events.metadata['dataset']
        
        jetHT = ak.sum( events["Jet"].pt, axis=1 )
        # print(jetHT[jetHT>950.])
        
        self.hists['ht'].fill( dataset=dataset, HT=jetHT )
        self.hists['htfull'].fill( dataset=dataset, HT=jetHT )
        
        return self.hists

    
    def postprocess(self, accumulator):
        return accumulator
    
    
    

In [ ]:
run = processor.Runner(
    executor = processor.FuturesExecutor(compression=None, workers=2),
    schema=NanoAODSchema,
    chunksize=100000,
    maxchunks=None
)

output = run(
    fileset,
    "Events",
    processor_instance=GetHT(),
)

In [ ]:
output['ht'].project("HT").plot()

In [ ]:
output['htfull'].project("HT").plot()

In [ ]:
# HThist = output['ht']
# HThist.axes['HT'].edges

HThist = output['htfull']
HThist.axes['HT'].edges

In [ ]:
# print('HT > ', HThist.axes['HT'].edges[0], '\tCounts = ', np.sum(HThist.values()[0][0:]))
# print('HT > ', HThist.axes['HT'].edges[45], '\tCounts = ', np.sum(HThist.values()[0][45:]))
# print('ratio = ', np.sum(HThist.values()[0][45:])/np.sum(HThist.values()[0][0:]))

print('HT > ', HThist.axes['HT'].edges[0], '\tCounts = ', np.sum(HThist.values()[0][0:]))
print('HT > ', HThist.axes['HT'].edges[9], '\tCounts = ', np.sum(HThist.values()[0][9:]))
print('ratio = ', np.sum(HThist.values()[0][9:])/np.sum(HThist.values()[0][0:]))